# P-Task3: Project Directory Auditor

**Problem:** DevOps team ko aisa tool chahiye jo project folder scan kare, har file ka name, size, type aur
last modified date nikaale aur timestamped audit report bana de. 30 din se purani files flag hon aur
overage fines ki alag penalties report bane. Scanning ka code custom module mein ho.

**Concepts used:** `os` module, `datetime` / `timedelta` / `strftime`, `math.ceil`,
apna module (`file_scanner.py`) import karna, `__name__ == "__main__"` guard.

**Fine rule (meri assumption):** file 30 din tak theek hai. Uske baad har extra din ka Rs 10.

## 1. Sample project folder banana
Folder aur files banayi, phir `os.utime()` se unki modified date purani kar di taake test ho sake.

In [1]:
import os
import time
from datetime import datetime, timedelta

import file_scanner          # mera apna module

root = "sample_project"
files = {                                   # file : kitne din purani
    "main.py": 2,
    "utils.py": 15,
    "config.json": 31,
    "README.md": 45,
    "old_backup.zip": 120,
    "data/students.csv": 10,
    "data/archive_2025.csv": 200,
    "docs/notes.txt": 60,
    "LICENSE": 29,
}

for rel_path, days_old in files.items():
    full = os.path.join(root, rel_path)
    os.makedirs(os.path.dirname(full), exist_ok=True)
    with open(full, "w") as f:
        f.write("sample data " * (len(rel_path) * 10))
    old_time = time.time() - days_old * 86400 - 3600      # +1 ghanta, taake din ka boundary saaf rahe
    os.utime(full, (old_time, old_time))

print("Sample files bani:", len(files))

Sample files bani: 9


## 2. Module se scan karwana

In [2]:
records = file_scanner.scan_folder(root)
now = datetime.now()

print(f"{'File':<22}{'Type':<10}{'Size KB':>8}  {'Modified':<18}{'Age(days)':>9}")
print("-" * 70)
for r in records:
    age = file_scanner.age_in_days(r["modified"], now)
    print(f"{r['name']:<22}{r['type']:<10}{file_scanner.size_kb(r['size_bytes']):>8}  "
          f"{r['modified']:%Y-%m-%d %H:%M}  {age:>8}")

File                  Type       Size KB  Modified          Age(days)
----------------------------------------------------------------------
LICENSE               (no ext)      0.82  2026-09-25 10:23        29
README.md             .md           1.05  2026-09-09 10:23        45
config.json           .json         1.29  2026-09-23 10:23        31
archive_2025.csv      .csv          2.46  2026-04-07 10:23       200
students.csv          .csv          1.99  2026-10-14 10:23        10
notes.txt             .txt          1.64  2026-08-25 10:23        60
main.py               .py           0.82  2026-10-22 10:23         2
old_backup.zip        .zip          1.64  2026-06-26 10:23       120
utils.py              .py           0.94  2026-10-09 10:23        15


## 3. Stale files (30 din se purani) aur fines

In [3]:
stale = [r for r in records if file_scanner.is_stale(r["modified"], now)]
penalties = []
for r in stale:
    fine = file_scanner.calculate_fine(r["modified"], now)
    penalties.append((r["name"], file_scanner.age_in_days(r["modified"], now), fine))

print("Total files :", len(records))
print("Stale files :", len(stale))
print()
for name, age, fine in sorted(penalties, key=lambda p: p[2], reverse=True):
    print(f"{name:<22} {age:>4} din purani   fine Rs {fine}")
print("\nTotal fine: Rs", sum(p[2] for p in penalties))

Total files : 9
Stale files : 5

archive_2025.csv        200 din purani   fine Rs 1710
old_backup.zip          120 din purani   fine Rs 910
notes.txt                60 din purani   fine Rs 310
README.md                45 din purani   fine Rs 160
config.json              31 din purani   fine Rs 20

Total fine: Rs 3110


## 4. Timestamped reports likhna
File ke naam mein date aur time lagaya (`strftime`) taake har baar nayi report bane, purani overwrite na ho.

In [4]:
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
os.makedirs("reports", exist_ok=True)

audit_path = os.path.join("reports", f"audit_report_{stamp}.txt")
with open(audit_path, "w") as f:
    f.write(f"AUDIT REPORT  -  {datetime.now():%Y-%m-%d %H:%M:%S}\n")
    f.write(f"Folder: {root}\n\n")
    for r in records:
        flag = "  <-- STALE" if file_scanner.is_stale(r["modified"], now) else ""
        f.write(f"{r['name']:<22}{r['type']:<10}{file_scanner.size_kb(r['size_bytes']):>7} KB  "
                f"{r['modified']:%Y-%m-%d}{flag}\n")
    f.write(f"\nTotal files: {len(records)}   Stale: {len(stale)}\n")

penalty_path = os.path.join("reports", f"penalties_report_{stamp}.txt")
with open(penalty_path, "w") as f:
    f.write(f"PENALTIES REPORT  -  {datetime.now():%Y-%m-%d %H:%M:%S}\n")
    f.write(f"Rule: 30 din ke baad Rs {file_scanner.FINE_PER_DAY} per din\n\n")
    for name, age, fine in penalties:
        f.write(f"{name:<22}{age:>4} din   Rs {fine}\n")
    f.write(f"\nTOTAL: Rs {sum(p[2] for p in penalties)}\n")

print(audit_path)
print(penalty_path)

reports/audit_report_20261024_112305.txt
reports/penalties_report_20261024_112305.txt


In [5]:
print(open(penalty_path).read())

PENALTIES REPORT  -  2026-10-24 11:23:05
Rule: 30 din ke baad Rs 10 per din

README.md               45 din   Rs 160
config.json             31 din   Rs 20
archive_2025.csv       200 din   Rs 1710
notes.txt               60 din   Rs 310
old_backup.zip         120 din   Rs 910

TOTAL: Rs 3110



## Conclusion
Scanning ka saara logic `file_scanner.py` mein hai aur notebook sirf usay import karke use karta hai.
9 mein se 5 files 30 din se purani nikli aur un ka total fine upar print hua. Reports `reports/` folder mein
timestamp ke saath save hui hain.